# 02d · G1: GoogLeNet transfer learning (Anudeep K.S. et al., Grenze 2024)

What the paper states, and how we reproduce it:

| Paper | Here |
|---|---|
| GoogLeNet, transfer learning | torchvision GoogLeNet, ImageNet weights |
| fine-tune the last three layers | `inception5a`, `inception5b`, `fc` trainable; everything else frozen |
| 224x224 input, softmax classifier, cross-entropy | same |
| 30 epochs | 30 epochs x 5,000 samples = 150k samples, the same training budget as V2 (drawn with the dataset's copy weighting; byte-identical copies add no information) |
| optimiser / learning rate not stated | Adam, lr 1e-4, constant (our choice, stated in the paper) |
| no augmentation mentioned | none |
| 80/20 split | our group-aware folds (same for every model) |

Checkpoint chosen by validation accuracy. Time: about 3 min per fold. Finished folds are skipped on re-run.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd

from common import config, data, env
from common.cv import run_cv
from common.train import TrainConfig

config.ensure_dirs()
device = env.require_cuda()
env.seed_everything()
uniq, splits = data.load_unique(), data.load_splits()
images = data.load_images(uniq)

In [ ]:
G1 = TrainConfig(
    name="G1", arch="googlenet", size=224, train_aug="plain", sampler="copies", samples_per_epoch=5000, epochs=30,
    batch_size=64, optimizer="adam", lr=1e-4, weight_decay=0.0, backbone_lr_mult=1.0, scheduler="constant",
    select="val_acc", notes="Anudeep K.S. et al.: GoogLeNet TL, last three layers fine-tuned, 30 epochs.")
summary = run_cv(G1, device, uniq, splits, images)

In [ ]:
pd.DataFrame({k: {"best_epoch": v["best_epoch"], "minutes": round(v["train_minutes"], 1),
                  "test_acc": v["test"]["accuracy"], "test_macro_f1": v["test"]["macro_f1"]}
              for k, v in summary["folds"].items()}).T